# HW01-A — Dockerized Airbnb Ops Package

Your first job is to make a tiny data package that can run the same way twice.

That sounds boring. Boring reproducibility is how production works.

By the end, you should have:

- a Python package
- a CLI command
- a Docker image
- a Docker Compose service
- a DVC stage
- a processed CSV
- a short run report

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials (for homeworks B & C, not this one). Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

Use docs when stuck. Guessing is slower.

- Python `pyproject.toml`: https://packaging.python.org/en/latest/guides/writing-pyproject-toml/
- Python CLI entry points: https://setuptools.pypa.io/en/latest/userguide/entry_point.html
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker Compose: https://docs.docker.com/compose/
- Docker volumes: https://docs.docker.com/engine/storage/volumes/
- DVC `dvc.yaml`: https://doc.dvc.org/user-guide/project-structure/dvcyaml-files
- DVC `repro`: https://doc.dvc.org/command-reference/repro

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

In [10]:
from pathlib import Path
import textwrap
import pandas as pd

PROJECT = Path.cwd()
for path in ['src/airbnb_ops', 'data/raw', 'data/processed', 'reports', 'tests']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_ops/__init__.py').write_text('__version__ = "0.1.0"\n')
print(PROJECT)

e:\MLOps\Week_01\HW01_A


## 1. Raw inputs

The two cells below are given to you.

Do not change these rows. The point of HW01-A is packaging, Docker, DVC, and validation, not inventing toy data.

Files created:

```text
data/raw/listings_sample.csv
data/raw/neighbourhood_segments.csv
```

The listings file includes `host_name` and `host_id` on purpose. Your PII code must handle them.


In [11]:
# Provided starter data 1.1
# Do not edit this cell.
# Implementation note:
# - This creates the listing-level raw input.
# - The host_name and host_id columns are intentionally included.
# - Later, your PII code must drop host_name and replace host_id with a stable host_key.

listings = pd.DataFrame(
    [
        {
            "listing_id": 1,
            "neighbourhood": "Centrum-West",
            "price": 180,
            "minimum_nights": 2,
            "availability_365": 120,
            "number_of_reviews": 31,
            "host_id": 901,
            "host_name": "Alice",
        },
        {
            "listing_id": 2,
            "neighbourhood": "Centrum-West",
            "price": 210,
            "minimum_nights": 3,
            "availability_365": 80,
            "number_of_reviews": 12,
            "host_id": 902,
            "host_name": "Bob",
        },
        {
            "listing_id": 3,
            "neighbourhood": "De Pijp",
            "price": 135,
            "minimum_nights": 2,
            "availability_365": 210,
            "number_of_reviews": 44,
            "host_id": 903,
            "host_name": "Chris",
        },
        {
            "listing_id": 4,
            "neighbourhood": "Oud-Noord",
            "price": 95,
            "minimum_nights": 1,
            "availability_365": 300,
            "number_of_reviews": 9,
            "host_id": 904,
            "host_name": "Dana",
        },
        {
            "listing_id": 5,
            "neighbourhood": "Oud-Noord",
            "price": 105,
            "minimum_nights": 2,
            "availability_365": 260,
            "number_of_reviews": 18,
            "host_id": 905,
            "host_name": "Eve",
        },
        {
            "listing_id": 6,
            "neighbourhood": "De Baarsjes",
            "price": 125,
            "minimum_nights": 3,
            "availability_365": 170,
            "number_of_reviews": 27,
            "host_id": 906,
            "host_name": "Farid",
        },
        {
            "listing_id": 7,
            "neighbourhood": "De Baarsjes",
            "price": 145,
            "minimum_nights": 4,
            "availability_365": 90,
            "number_of_reviews": 21,
            "host_id": 907,
            "host_name": "Grace",
        },
        {
            "listing_id": 8,
            "neighbourhood": "Westerpark",
            "price": 155,
            "minimum_nights": 2,
            "availability_365": 140,
            "number_of_reviews": 36,
            "host_id": 908,
            "host_name": "Hamed",
        },
    ]
)

listings.to_csv(PROJECT / "data/raw/listings_sample.csv", index=False)
listings.head()


,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_id,host_name
0,1,Centrum-West,180,2,120,31,901,Alice
1,2,Centrum-West,210,3,80,12,902,Bob
2,3,De Pijp,135,2,210,44,903,Chris
3,4,Oud-Noord,95,1,300,9,904,Dana
4,5,Oud-Noord,105,2,260,18,905,Eve


In [12]:
# Provided starter data 1.2
# Do not edit this cell.
# Implementation note:
# - This file enriches neighbourhoods with business metadata.
# - Your transform step should join this file after aggregation.
# - If a neighbourhood has no segment row, your code should fill it with "unknown".

segments = pd.DataFrame(
    [
        {
            "neighbourhood": "Centrum-West",
            "tourism_segment": "tourist-heavy",
            "priority_level": "high",
        },
        {
            "neighbourhood": "De Pijp",
            "tourism_segment": "mixed",
            "priority_level": "medium",
        },
        {
            "neighbourhood": "Oud-Noord",
            "tourism_segment": "emerging",
            "priority_level": "medium",
        },
        {
            "neighbourhood": "De Baarsjes",
            "tourism_segment": "local-heavy",
            "priority_level": "low",
        },
        {
            "neighbourhood": "Westerpark",
            "tourism_segment": "mixed",
            "priority_level": "medium",
        },
    ]
)

segments.to_csv(PROJECT / "data/raw/neighbourhood_segments.csv", index=False)
segments


,neighbourhood,tourism_segment,priority_level
0,Centrum-West,tourist-heavy,high
1,De Pijp,mixed,medium
2,Oud-Noord,emerging,medium
3,De Baarsjes,local-heavy,low
4,Westerpark,mixed,medium


In [13]:
# Checkpoint
for file in ['data/raw/listings_sample.csv', 'data/raw/neighbourhood_segments.csv']:
    assert Path(file).exists(), f'Missing {file}'

pd.read_csv('data/raw/listings_sample.csv').head()

,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_id,host_name
0,1,Centrum-West,180,2,120,31,901,Alice
1,2,Centrum-West,210,3,80,12,902,Bob
2,3,De Pijp,135,2,210,44,903,Chris
3,4,Oud-Noord,95,1,300,9,904,Dana
4,5,Oud-Noord,105,2,260,18,905,Eve


## 2. Data contract

Write code against a contract, not vibes.

Input listing columns:

```text
listing_id, neighbourhood, price, minimum_nights,
availability_365, number_of_reviews, host_id, host_name
```

Output columns:

```text
neighbourhood, num_listings, avg_price, median_price,
avg_minimum_nights, availability_365_avg, total_reviews,
reviews_per_listing, tourism_segment, priority_level
```

In [ ]:
# TODO 2.1
# Create src/airbnb_ops/config.py.
# Add a PipelineConfig dataclass with default paths for:
# listings_path, segments_path, output_path, report_path

# Write your code here.

####################### Create File 
from pathlib import Path

config_py = '''from dataclasses import dataclass
from pathlib import Path

@dataclass
class PipelineConfig:
    listings_path: Path = Path("data/raw/listings_sample.csv")
    segments_path: Path = Path("data/raw/neighbourhood_segments.csv")
    output_path: Path = Path("data/processed/neighbourhood_summary.csv")
    report_path: Path = Path("reports/neighbourhood_report.md")
'''

config_path = PROJECT / "src/airbnb_ops/config.py"

config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(config_py, encoding="utf-8")

####################### Reload to apply changes
import importlib
import src.airbnb_ops.config

importlib.invalidate_caches()
importlib.reload(src.airbnb_ops.config)


####################### Call the code
from src.airbnb_ops.config import PipelineConfig

config = PipelineConfig()

listings_path = config.listings_path
segments_path = config.segments_path
output_path = config.output_path
report_path = config.report_path

In [ ]:
# TODO 2.2
# Create src/airbnb_ops/extract.py.
# Add read_csv_checked(path: Path) -> pd.DataFrame.
# It should raise FileNotFoundError if the file is missing.

# Write your code here.

####################### Create File 
# Create or overwrite the config.py file.
# This file stores the default input, output, and report paths used by the pipeline.

from pathlib import Path

extract_py = '''
from pathlib import Path
import pandas as pd

def read_csv_checked(path: Path) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    return pd.read_csv(path)
'''

extract_path = PROJECT / "src/airbnb_ops/extract.py"

extract_path.parent.mkdir(parents=True, exist_ok=True)
extract_path.write_text(extract_py, encoding="utf-8")

####################### Reload to apply changes
import importlib
import src.airbnb_ops.extract

importlib.invalidate_caches()
importlib.reload(src.airbnb_ops.extract)


####################### Call the code
from src.airbnb_ops.extract import read_csv_checked

listings_df = read_csv_checked(listings_path)

## 3. PII handling

For this homework:

- drop `host_name`
- convert `host_id` to `host_key`
- drop the original `host_id`

Do not use Python's built-in `hash()`. It is not stable across sessions. Use `hashlib.sha256`.

In [ ]:
# TODO 3.1
# Create src/airbnb_ops/pii.py.
# Add:
# - DIRECT_PII_COLUMNS
# - pseudonymize_value(value, salt='qbc12')
# - handle_pii(df)

# Write your code here.

####################### Create File 
# This module removes direct PII columns and creates a hashed host_key from host_id.

from pathlib import Path

pii_py = '''
import hashlib

DIRECT_PII_COLUMNS = ["host_name"]

def pseudonymize_value(value, salt='qbc12'):
    new_value = f"{salt}:{value}"
    code = hashlib.sha256(new_value.encode("utf-8")).hexdigest()
    return code

def handle_pii(df):

    df = df.copy()
    df = df.drop(columns=DIRECT_PII_COLUMNS, errors="ignore")

    df["host_key"] = df["host_id"].apply(pseudonymize_value)
    df = df.drop(columns=["host_id"])

    return df
'''

pii_path = PROJECT / "src/airbnb_ops/pii.py"

pii_path.parent.mkdir(parents=True, exist_ok=True)
pii_path.write_text(pii_py, encoding="utf-8")

####################### Reload to apply changes
import importlib
import src.airbnb_ops.pii

importlib.invalidate_caches()
importlib.reload(src.airbnb_ops.pii)


####################### Call the code
from src.airbnb_ops.pii import handle_pii

safe_listings_df = handle_pii(listings_df)
safe_listings_df.head()

,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_key
0,1,Centrum-West,180,2,120,31,157b469a56bea3d8d61f8289988c48277d41246ee70edd...
1,2,Centrum-West,210,3,80,12,21abc69946417e590e593249bb58a1ebcf5a2c936e528d...
2,3,De Pijp,135,2,210,44,787c7dcd2e488c99341b4db880069c8db64705a280e69f...
3,4,Oud-Noord,95,1,300,9,dc29735cbec8fb0500ef68b24c729185a1beaf51ed4e2d...
4,5,Oud-Noord,105,2,260,18,29bbea2a5bd02fb08e5c4bd9c03c13414d1cfa2de8d0c8...


## 4. Transform

Build a neighbourhood-level summary. Use `groupby`, not loops.

Join the segment metadata after aggregation. If a segment is missing, fill it with `unknown`.

In [ ]:
# TODO 4.1
# Create src/airbnb_ops/transform.py.
# Add build_neighbourhood_summary(listings, segments).
# It should validate required input columns, aggregate, join segments, and return a dataframe.

# Write your code here.

####################### Create File 

'''
This module validates required columns, aggregates listings by neighbourhood,
joins neighbourhood segment metadata, and returns the final output dataframe with required columns.
Output columns:
neighbourhood, num_listings, avg_price, median_price,
avg_minimum_nights, availability_365_avg, total_reviews,
reviews_per_listing, tourism_segment, priority_level
'''

from pathlib import Path

transform_py = '''import pandas as pd


REQUIRED_LISTING_COLUMNS = [
    "listing_id",
    "neighbourhood",
    "price",
    "minimum_nights",
    "availability_365",
    "number_of_reviews",
]

REQUIRED_SEGMENT_COLUMNS = [
    "neighbourhood",
    "tourism_segment",
    "priority_level",
]

OUTPUT_COLUMNS = [
    "neighbourhood",
    "num_listings",
    "avg_price",
    "median_price",
    "avg_minimum_nights",
    "availability_365_avg",
    "total_reviews",
    "reviews_per_listing",
    "tourism_segment",
    "priority_level",
]


def build_neighbourhood_summary(listings, segments):
    missing_listing_cols = set(REQUIRED_LISTING_COLUMNS) - set(listings.columns)
    if missing_listing_cols:
        raise ValueError(f"Missing listing columns: {missing_listing_cols}")

    missing_segment_cols = set(REQUIRED_SEGMENT_COLUMNS) - set(segments.columns)
    if missing_segment_cols:
        raise ValueError(f"Missing segment columns: {missing_segment_cols}")
    
    summary = listings.groupby("neighbourhood").agg(num_listings=("listing_id", "count"),
                                                    avg_price=("price", "mean"),
                                                    median_price=("price", "median"),
                                                    avg_minimum_nights=("minimum_nights", "mean"),
                                                    availability_365_avg=("availability_365", "mean"),
                                                    total_reviews=("number_of_reviews", "sum"),
                                                    ).reset_index()
    
    summary["reviews_per_listing"] =  (summary["total_reviews"] / summary["num_listings"])

    summary = summary.merge(
        segments,
        on="neighbourhood",
        how="left",
    )

    summary[["tourism_segment", "priority_level"]] = summary[
        ["tourism_segment", "priority_level"]
    ].fillna("unknown")

    return summary[OUTPUT_COLUMNS]
'''

transform_path = PROJECT / "src/airbnb_ops/transform.py"

transform_path.parent.mkdir(parents=True, exist_ok=True)
transform_path.write_text(transform_py, encoding="utf-8")

####################### Reload to apply changes
import importlib
import src.airbnb_ops.transform

importlib.invalidate_caches()
importlib.reload(src.airbnb_ops.transform)


####################### Call the code
from src.airbnb_ops.transform import build_neighbourhood_summary

summary_df = build_neighbourhood_summary(safe_listings_df, segments)

summary_df

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,availability_365_avg,total_reviews,reviews_per_listing,tourism_segment,priority_level
0,Centrum-West,2,195.0,195.0,2.5,100.0,43,21.5,tourist-heavy,high
1,De Baarsjes,2,135.0,135.0,3.5,130.0,48,24.0,local-heavy,low
2,De Pijp,1,135.0,135.0,2.0,210.0,44,44.0,mixed,medium
3,Oud-Noord,2,100.0,100.0,1.5,280.0,27,13.5,emerging,medium
4,Westerpark,1,155.0,155.0,2.0,140.0,36,36.0,mixed,medium


## 5. Validation

Validation is the bouncer. Bad output does not get into the club.

Minimum checks:

- output is not empty
- required output columns exist
- no PII columns exist
- neighbourhood is not null
- num_listings > 0
- avg_price >= 0
- availability_365_avg between 0 and 365

In [29]:
# TODO 5.1
# Create src/airbnb_ops/validate.py.
# Add validate_summary(summary) -> None.
# Raise ValueError when a check fails.

# Write your code here.

####################### Create File 

from pathlib import Path

validate_py = '''import pandas as pd

REQUIRED_OUTPUT_COLUMNS = [
    "neighbourhood",
    "num_listings",
    "avg_price",
    "median_price",
    "avg_minimum_nights",
    "availability_365_avg",
    "total_reviews",
    "reviews_per_listing",
    "tourism_segment",
    "priority_level",
]

PII_COLUMNS = [
    "host_name",
    "host_id",
]


def validate_summary(summary: pd.DataFrame) -> None:
    if summary.empty:
        raise ValueError("Summary output is empty")

    missing_columns = set(REQUIRED_OUTPUT_COLUMNS) - set(summary.columns)
    if missing_columns:
        raise ValueError(f"Missing required output columns: {missing_columns}")

    pii_columns_found = set(PII_COLUMNS) & set(summary.columns)
    if pii_columns_found:
        raise ValueError(f"PII columns found in output: {pii_columns_found}")

    if summary["neighbourhood"].isna().any():
        raise ValueError("neighbourhood contains null values")

    if (summary["num_listings"] <= 0).any():
        raise ValueError("num_listings must be greater than 0")

    if (summary["avg_price"] < 0).any():
        raise ValueError("avg_price must be greater than or equal to 0")

    if (~summary["availability_365_avg"].between(0, 365)).any():
        raise ValueError("availability_365_avg must be between 0 and 365")
    
'''

validate_path = PROJECT / "src/airbnb_ops/validate.py"

validate_path.parent.mkdir(parents=True, exist_ok=True)
validate_path.write_text(validate_py, encoding="utf-8")

####################### Reload to apply changes
import importlib
import src.airbnb_ops.validate

importlib.invalidate_caches()
importlib.reload(src.airbnb_ops.validate)


####################### Call the code
from src.airbnb_ops.validate import validate_summary

validate_summary(summary_df)

## 6. CLI

The package should expose one command:

```bash
airbnb-ops run
```

The command should read inputs, handle PII, transform, validate, write CSV, and write a markdown report.

In [42]:
# TODO 6.1
# Create src/airbnb_ops/cli.py using Typer.
# Add a run command.
# It should write:
# - data/processed/airbnb_neighbourhood_summary.csv
# - reports/hw01_a_run_report.md

# Write your code here.


####################### Create File 

from pathlib import Path

cli_py = '''from pathlib import Path

import typer

from airbnb_ops.config import PipelineConfig
from airbnb_ops.extract import read_csv_checked
from airbnb_ops.pii import handle_pii
from airbnb_ops.transform import build_neighbourhood_summary
from airbnb_ops.validate import validate_summary


app = typer.Typer()


@app.callback()
def main() -> None:
    pass


@app.command("run")
def run_pipeline() -> None:
    config = PipelineConfig(
        output_path=Path("data/processed/airbnb_neighbourhood_summary.csv"),
        report_path=Path("reports/hw01_a_run_report.md"),
    )

    listings = read_csv_checked(config.listings_path)
    segments = read_csv_checked(config.segments_path)

    safe_listings = handle_pii(listings)
    summary = build_neighbourhood_summary(safe_listings, segments)

    validate_summary(summary)

    config.output_path.parent.mkdir(parents=True, exist_ok=True)
    config.report_path.parent.mkdir(parents=True, exist_ok=True)

    summary.to_csv(config.output_path, index=False)

    report = f"""# Airbnb Ops Run Report

                ## Status

                Run completed successfully.

                ## Outputs

                - Summary CSV: `{config.output_path}`
                - Report: `{config.report_path}`

                ## Rows written

                {len(summary)}

                ## Columns

                {", ".join(summary.columns)}
                """

    config.report_path.write_text(report, encoding="utf-8")

    typer.echo(f"Wrote summary to {config.output_path}")
    typer.echo(f"Wrote report to {config.report_path}")


if __name__ == "__main__":
    app()
    
'''

cli_path = PROJECT / "src/airbnb_ops/cli.py"

cli_path.parent.mkdir(parents=True, exist_ok=True)
cli_path.write_text(cli_py, encoding="utf-8")


1627

## 7. Package metadata

`pyproject.toml` makes the project installable. `[project.scripts]` creates the `airbnb-ops` command.

In [44]:
# TODO 7.1
# Create pyproject.toml.
# Requirements:
# - project name: airbnb-ops
# - dependencies: pandas, typer, rich
# - script: airbnb-ops = airbnb_ops.cli:app

# Write your code here.


from pathlib import Path

pyproject_txt = '''[project]
name = "airbnb-ops"
version = "0.1.0"
description = "Airbnb neighbourhood operations pipeline"
requires-python = ">=3.9"

dependencies = [
    "pandas",
    "typer",
    "rich"
]

[project.scripts]
airbnb-ops = "airbnb_ops.cli:app"

[tool.setuptools.packages.find]
where = ["src"]

[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"
'''

pyproject_path = PROJECT / "pyproject.toml"

pyproject_path.write_text(pyproject_txt, encoding="utf-8")


377

In [ ]:
# TODO 7.2
# Create requirements.txt with pandas, typer, rich, dvc.

# Write your code here.


from pathlib import Path

requirements_txt = '''pandas
typer
rich
dvc
'''

requirements_path = PROJECT / "requirements.txt"

requirements_path.write_text(requirements_txt, encoding="utf-8")

############## Then I run following in terminal
# pip install -r requirements.txt

e:\MLOps\Week_01\HW01_A\requirements.txt


In [43]:
# Local smoke test
import sys

!{sys.executable} -m pip install -q -e . --no-build-isolation -i https://pypi.devneeds.ir/simple/ --trusted-host pypi.devneeds.ir
!airbnb-ops run
pd.read_csv('data/processed/airbnb_neighbourhood_summary.csv').head()

    tinycss2 (>=1.1.0<1.2) ; extra == 'css'
             ~~~~~~~~^


Wrote summary to data\processed\airbnb_neighbourhood_summary.csv
Wrote report to reports\hw01_a_run_report.md


,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,availability_365_avg,total_reviews,reviews_per_listing,tourism_segment,priority_level
0,Centrum-West,2,195.0,195.0,2.5,100.0,43,21.5,tourist-heavy,high
1,De Baarsjes,2,135.0,135.0,3.5,130.0,48,24.0,local-heavy,low
2,De Pijp,1,135.0,135.0,2.0,210.0,44,44.0,mixed,medium
3,Oud-Noord,2,100.0,100.0,1.5,280.0,27,13.5,emerging,medium
4,Westerpark,1,155.0,155.0,2.0,140.0,36,36.0,mixed,medium


## 8. Docker

Create `.dockerignore`, `Dockerfile`, and `docker-compose.yml`.

Watch out for this common garbage move: copying generated outputs into the image. Do not do that. Mount `data/` and `reports/` so the container writes outputs to your working directory.

In [53]:
# TODO 8.1
# Create .dockerignore.
# Exclude .git, .venv, __pycache__, .ipynb_checkpoints, data/processed, reports.

# Write your code here.

from pathlib import Path

docker_ignore_content = """
.git
.venv
__pycache__
.ipynb_checkpoint

data/preprocessed
reports

"""

Path(PROJECT / ".dockerignore").write_text(docker_ignore_content ,encoding="utf-8")

70

In [54]:
# TODO 8.2
# Create Dockerfile.
# Requirements:
# - FROM python:3.11-slim
# - WORKDIR /app
# - copy package metadata and src/
# - install requirements and package
# - default command: airbnb-ops run

# Write your code here.

from pathlib import Path

dockerfile_content = """\
FROM python:3.11-slim

WORKDIR /app

COPY pyproject.toml requirements.txt ./
COPY src/ ./src/

RUN pip install --no-cache-dir -r requirements.txt
RUN pip install --no-cache-dir .

CMD ["airbnb-ops", "run"]
"""

Path(PROJECT / "Dockerfile").write_text(dockerfile_content, encoding="utf-8")


206

In [55]:
# TODO 8.3
# Create docker-compose.yml.
# Requirements:
# - service name: airbnb-ops
# - build current directory
# - mount ./data and ./reports
# - command: airbnb-ops run

# Write your code here.

from pathlib import Path

compose_content = """\
services:
  airbnb-ops:
    build: .
    volumes:
      - ./data:/app/data
      - ./reports:/app/reports
    command: airbnb-ops run
"""

Path(PROJECT / "docker-compose.yml").write_text(compose_content, encoding="utf-8")

134

In [49]:
# Docker smoke test. Run in terminal if notebook cannot access Docker.
!docker compose build
print("yes")
!docker compose run --rm airbnb-ops

yes
#1 [internal] load local bake definitions
#1 reading from stdin 520B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 256B done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 DONE 1.9s

#4 [1/6] FROM docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0
#4 resolve docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0 0.0s done
#4 DONE 0.0s

#5 [2/6] WORKDIR /app
#5 CACHED

#6 [3/6] COPY pyproject.toml requirements.txt ./
#6 DONE 0.0s

#7 [4/6] COPY src/ ./src/
#7 DONE 0.0s

#8 [internal] load .dockerignore
#8 transferring context: 119B done
#8 DONE 0.0s

#9 [internal] load build context
#9 transferring context: 15.25kB done
#9 DONE 0.0s

#10 [5/6] RUN pip install --no-cache-dir -r requirements.txt
#10 4.895 Collecting pandas (from -r requirements.txt (line 1))
#10 20.69   Downloading pandas-3

failed to solve: process "/bin/sh -c pip install --no-cache-dir -r requirements.txt" did not complete successfully: exit code: 2



Wrote summary to data/processed/airbnb_neighbourhood_summary.csv
Wrote report to reports/hw01_a_run_report.md


## 9. DVC

A DVC stage is the receipt for how your output was produced.

Create one stage named `run_pipeline`.

In [56]:
# TODO 9.1
# Create dvc.yaml.
# Stage:
# - cmd: airbnb-ops run
# - deps: src/airbnb_ops, both raw CSV files
# - outs: processed CSV
# - metrics/report: reports/hw01_a_run_report.md with cache: false

# Write your code here.

from pathlib import Path

dvc_yaml_content = """\
stages:
  run_pipeline:
    cmd: airbnb-ops run
    deps:
      - src/airbnb_ops
      - data/raw/listings_sample.csv
      - data/raw/neighbourhood_segments.csv
    outs:
      - data/processed/airbnb_neighbourhood_summary.csv
    metrics:
      - reports/hw01_a_run_report.md:
          cache: false
"""

Path(PROJECT / "dvc.yaml").write_text(dvc_yaml_content, encoding="utf-8")

302

In [51]:
# Extra credit 9.2
# Run these commands if DVC is installed in your local environment.
# This is a useful check, but the required part is creating a correct dvc.yaml file.

!dvc repro
!dvc dag


Running stage 'run_pipeline':
> airbnb-ops run
Wrote summary to data\processed\airbnb_neighbourhood_summary.csv
Wrote report to reports\hw01_a_run_report.md
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.
+--------------+ 
| run_pipeline | 
+--------------+ 


## Final proof

If this cell fails, HW01-A is not done.

In [52]:
output = Path('data/processed/airbnb_neighbourhood_summary.csv')
report = Path('reports/hw01_a_run_report.md')
assert output.exists(), 'Output file was not created.'
assert report.exists(), 'Report file was not created.'

df = pd.read_csv(output)
required = {'neighbourhood','num_listings','avg_price','median_price','avg_minimum_nights','availability_365_avg','total_reviews','reviews_per_listing','tourism_segment','priority_level'}
assert not (required - set(df.columns)), f'Missing: {required - set(df.columns)}'
for bad in ['host_name','host_id','reviewer_name','reviewer_id','listing_url','host_url']:
    assert bad not in df.columns, f'PII leaked: {bad}'
df.head()

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,availability_365_avg,total_reviews,reviews_per_listing,tourism_segment,priority_level
0,Centrum-West,2,195.0,195.0,2.5,100.0,43,21.5,tourist-heavy,high
1,De Baarsjes,2,135.0,135.0,3.5,130.0,48,24.0,local-heavy,low
2,De Pijp,1,135.0,135.0,2.0,210.0,44,44.0,mixed,medium
3,Oud-Noord,2,100.0,100.0,1.5,280.0,27,13.5,emerging,medium
4,Westerpark,1,155.0,155.0,2.0,140.0,36,36.0,mixed,medium


## Commit

```bash
git add .
git commit -m "HW01-A dockerized Airbnb package"
```